# Hierarchically Organizing Data

Similar to parallel arrays, RockVerse provides classes based on Zarr groups 
to help organize data hierarchically in complex datasets.

## The Group Class

RockVerse’s Group class is a high-level interface for managing Zarr groups
across MPI processes, ensuring synchronized access and consistent metadata 
management.

If you're unfamiliar with Zarr groups, we recommend exploring the
[Zarr user guide for groups](https://zarr.readthedocs.io/en/stable/user-guide/groups.html)
to gain a deeper understanding of its fundamentals.

You can create a new group at a specified storage location and path using the
`create_group` function:

In [1]:
import numpy as np
import rockverse as rv
group = rv.create_group(store='/path/to/store', path='my_group', overwrite=True)
print(group)

This will create a [Group](../../../api/core/group.rst) object.
Once created, you can add subgroups and arrays within the group. For example:

In [2]:
# Create a parallel array in the group
array1 = group.create_array('temperature', shape=(100, 100), dtype='float64', chunks=(10, 10))

# Create a coordinate in the group
coord0 = group.create_coordinate(path='x', data=np.arange(100, dtype=float))

# Create a subgroup
subgroup = group.create_group('subgroup_path')

# Subgroups are just other groups
# Create a parallel array in the new subgroup
array2 = subgroup.create_array('pressure', shape=(100, 100), dtype='float64', chunks=(10, 10))

Parent groups will be automatically created as needed when you provide a nested directory path to the creation functions:

In [3]:
# This will create 'foo' and 'bar' groups and add 'ct_attenuation' under 'bar'
ct = group.create_array('foo/bar/ct_attenuation', shape=(100, 100), dtype='float64', chunks=(10, 10))
print(ct)

You can access members of a group by key. It’s also possible to use the full path within 
the group as the key, using a slash (`/`) as a separator:

In [4]:
print('x:', group['x'])
print('Temperature:', group['temperature'])
print('Pressure:', group['subgroup_path'])
print('Pressure:', group['subgroup_path/pressure'])
print('CT atteuation', group['foo/bar/ct_attenuation'])

x: <rockverse.core.coordinates.Coordinate object at 0x000001DFF020A270>
Temperature: <rockverse.core.parallelarray.ParallelArray object at 0x000001DFF020A270>
Pressure: <rockverse.core.group.Group object at 0x000001DFFE7A9D90>
Pressure: <rockverse.core.parallelarray.ParallelArray object at 0x000001DFFE77F080>
CT atteuation <rockverse.core.parallelarray.ParallelArray object at 0x000001DFFE77F080>


Groups also offer the `attrs` property, similar to parallel arrays, which handles MPI synchronization transparently under the hood:

In [5]:
group.attrs

Attributes({'_ROCKVERSE_DATATYPE': 'Group'})

In [6]:
group.attrs['description'] = 'Dataset from area XYZ'
print(group.attrs['description'])

Dataset from area XYZ


Direct item assignment to Group objects is not supported and will raise an error, for example:

```python
# THIS WILL RAISE AN ERROR
import numpy as np
group['another_array'] = rv.array(np.random.randn(10))
```

Use the provided creation functions (see the [API documentation](../../../api/core/group.rst))
to ensure proper MPI synchronization and metadata management.